# Supplementary Figure S1: spatial distribution of land-use data availability

Reviewer #1 requested a map to assess potential spatial bias associated with the 1,396 grid cells lacking readable land-use representative points within 5 km.

This notebook is designed to run **standalone in Google Colab**. It mounts Google Drive, locates the same grid-level GIS/domain-feature source used by the modelling notebooks, resolves auto-prefixed land-use column names, verifies the manuscript counts, and then draws the supplementary map.

Expected counts:
- land-use available: **4,095** grids
- land-use missing: **1,396** grids
- total: **5,491** grids


In [ ]:
# ============================================================
# 1. Setup, Google Drive mount, and source-file discovery
# ============================================================
!pip -q install geopandas pyogrio

import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# Mount Drive explicitly so this notebook works when run by itself.
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print('Google Drive is already visible at /content/drive/MyDrive')
except Exception as e:
    print('Drive mount skipped or failed:', repr(e))

BASE_DIR = Path('/content/drive/MyDrive/avian_influenza_project')
PROC_DIR = BASE_DIR / 'processed'
MODEL_DIR = PROC_DIR / 'model_outputs_riskmap_eval'
DOMAIN_DIR = PROC_DIR / 'domain_features'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# The 14_v5 modelling notebook actually used
# processed/model_outputs_riskmap_eval/14_grid_level_domain_features.csv
# in the previously executed analysis. Include that source explicitly.
DOMAIN_CANDIDATES = [
    DOMAIN_DIR / 'grid_environment_features.csv',
    DOMAIN_DIR / 'grid_environment_features.parquet',
    MODEL_DIR / '18_v8_grid_environment_features_for_14.csv',
    MODEL_DIR / '18_v8_grid_environment_features_for_14.parquet',
    MODEL_DIR / '14_grid_level_domain_features.csv',
    MODEL_DIR / '14_grid_level_domain_features.parquet',
    MODEL_DIR / '18_grid_environment_features.csv',
    MODEL_DIR / '18_grid_environment_features.parquet',
    BASE_DIR / 'domain_features' / 'grid_environment_features.csv',
    BASE_DIR / 'domain_features' / 'grid_environment_features.parquet',
]

def read_table(p):
    p = Path(p)
    if p.suffix.lower() == '.csv':
        return pd.read_csv(p)
    if p.suffix.lower() == '.parquet':
        return pd.read_parquet(p)
    raise ValueError(f'Unsupported file: {p}')

def has_required_structure(p):
    """Quickly test whether a candidate contains grid_id, coordinates, and land-use-like columns."""
    try:
        p = Path(p)
        if p.suffix.lower() == '.csv':
            h = pd.read_csv(p, nrows=5)
        elif p.suffix.lower() == '.parquet':
            h = pd.read_parquet(p).head(5)
        else:
            return False
        cols = [str(c).lower() for c in h.columns]
        has_grid = 'grid_id' in h.columns
        has_coord = (any(c in h.columns for c in ['grid_lat','centroid_lat','lat','latitude']) and
                     any(c in h.columns for c in ['grid_lon','centroid_lon','lon','longitude']))
        has_landuse = any(any(key in c for key in [
            'paddy_ratio_5km','farmland_ratio_5km','forest_ratio_5km',
            'urban_ratio_5km','waterbody_ratio_5km']) for c in cols)
        return has_grid and has_coord and has_landuse
    except Exception:
        return False

# First try known sources.
path = next((p for p in DOMAIN_CANDIDATES if p.exists() and has_required_structure(p)), None)

# Fallback: search likely feature files under processed/ without touching the huge weekly panel.
if path is None:
    patterns = [
        '**/*grid*environment*feature*.csv', '**/*grid*environment*feature*.parquet',
        '**/*grid*level*domain*feature*.csv', '**/*grid*level*domain*feature*.parquet',
        '**/*domain*feature*.csv', '**/*domain*feature*.parquet',
    ]
    discovered = []
    for pat in patterns:
        discovered.extend(Path(PROC_DIR).glob(pat))
    # Deduplicate while preserving order.
    discovered = list(dict.fromkeys(discovered))
    path = next((p for p in discovered if has_required_structure(p)), None)

if path is None:
    print('Checked known candidates:')
    for p in DOMAIN_CANDIDATES:
        print('  ', p, 'exists=', p.exists())
    raise FileNotFoundError(
        'Land-use grid feature file could not be located. '
        'Please confirm that notebook 18_v8 or the domain-feature construction notebook has been run.'
    )

print('Selected source:', path)
df = read_table(path).drop_duplicates('grid_id').copy()
print('Loaded shape:', df.shape)
print('Unique grid_id:', df['grid_id'].nunique())


In [ ]:
# ============================================================
# 2. Resolve coordinates and the five land-use indicators
#    (works with both plain and auto-prefixed column names)
# ============================================================

lat_col = next((c for c in ['grid_lat','centroid_lat','lat','latitude'] if c in df.columns), None)
lon_col = next((c for c in ['grid_lon','centroid_lon','lon','longitude'] if c in df.columns), None)
if lat_col is None or lon_col is None:
    raise ValueError('Latitude/longitude columns were not found in the selected source.')

df = df.rename(columns={lat_col:'grid_lat', lon_col:'grid_lon'})
df['grid_lat'] = pd.to_numeric(df['grid_lat'], errors='coerce')
df['grid_lon'] = pd.to_numeric(df['grid_lon'], errors='coerce')

CANONICAL = [
    'waterbody_ratio_5km',
    'urban_ratio_5km',
    'paddy_ratio_5km',
    'farmland_ratio_5km',
    'forest_ratio_5km',
]

def choose_feature_column(frame, canonical):
    # 1) Exact final column is preferred.
    if canonical in frame.columns:
        return canonical

    # 2) The real 18_v8 features were later merged into 14_grid_level_domain_features
    #    with the prefix 'domain_grid_environment_features_'. Prefer those if present.
    preferred = f'domain_grid_environment_features_{canonical}'
    if preferred in frame.columns:
        return preferred

    # 3) Otherwise accept auto-prefixed columns ending in the canonical name.
    #    Choose the candidate with the largest number of non-missing numeric values;
    #    this avoids selecting empty template columns.
    candidates = [c for c in frame.columns if str(c).lower().endswith(canonical.lower())]
    if not candidates:
        candidates = [c for c in frame.columns if canonical.lower() in str(c).lower()]
    if not candidates:
        return None

    scored = []
    for c in candidates:
        x = pd.to_numeric(frame[c], errors='coerce')
        scored.append((int(x.notna().sum()), c))
    scored.sort(reverse=True)
    return scored[0][1]

resolved = {name: choose_feature_column(df, name) for name in CANONICAL}
print('Resolved land-use columns:')
for k,v in resolved.items():
    print(f'  {k} <- {v}')

unresolved = [k for k,v in resolved.items() if v is None]
if unresolved:
    landuse_like = [c for c in df.columns if any(s in str(c).lower() for s in ['paddy','farmland','forest','urban','waterbody'])]
    print('Land-use-like columns found:')
    for c in landuse_like:
        print('  ', c)
    raise ValueError(f'Could not resolve these land-use indicators: {unresolved}')

# Copy to canonical names so downstream code is simple and transparent.
for canonical, source_col in resolved.items():
    df[canonical] = pd.to_numeric(df[source_col], errors='coerce')

# Show non-missing counts for each indicator.
counts = pd.DataFrame({
    'indicator': CANONICAL,
    'source_column': [resolved[c] for c in CANONICAL],
    'non_missing': [int(df[c].notna().sum()) for c in CANONICAL],
    'missing': [int(df[c].isna().sum()) for c in CANONICAL],
})
display(counts)

# The five indicators were constructed from the same readable land-use points,
# so their missingness masks should be identical. Verify rather than assume it.
mask_matrix = df[CANONICAL].notna()
mask_identical = all(mask_matrix[c].equals(mask_matrix[CANONICAL[0]]) for c in CANONICAL[1:])
print('Identical missingness mask across five indicators:', mask_identical)
if not mask_identical:
    print('WARNING: missingness masks differ; availability will require all five indicators to be present.')

df['landuse_available'] = df[CANONICAL].notna().all(axis=1)
df['landuse_status'] = np.where(df['landuse_available'], 'Available', 'Missing')

summary = df['landuse_status'].value_counts().reindex(['Available','Missing'], fill_value=0).rename_axis('status').reset_index(name='n_grids')
summary['percent'] = 100 * summary['n_grids'] / len(df)

display(summary)

expected_total = 5491
expected_available = 4095
expected_missing = 1396
actual_total = int(df['grid_id'].nunique())
actual_available = int(df['landuse_available'].sum())
actual_missing = int((~df['landuse_available']).sum())

print(f'Actual: total={actual_total}, available={actual_available}, missing={actual_missing}')
print(f'Expected manuscript counts: total={expected_total}, available={expected_available}, missing={expected_missing}')
print(
    'Grid-centroid coordinate range:',
    f"lon={df['grid_lon'].min():.2f}–{df['grid_lon'].max():.2f}, "
    f"lat={df['grid_lat'].min():.2f}–{df['grid_lat'].max():.2f}"
)
if not (
    df['grid_lon'].between(123.5, 149.0, inclusive='both').all()
    and df['grid_lat'].between(24.0, 46.0, inclusive='both').all()
):
    raise ValueError('At least one of the 5,491 grid centroids lies outside the full-extent axes.')

# Stop before creating a potentially inconsistent figure.
if (actual_total, actual_available, actual_missing) != (expected_total, expected_available, expected_missing):
    raise ValueError(
        'The selected source does not reproduce the manuscript counts '
        f'(actual {actual_total}/{actual_available}/{actual_missing}; '
        f'expected {expected_total}/{expected_available}/{expected_missing}). '
        'Do not use the map until the source is reconciled.'
    )

# Save audit files.
summary.to_csv(MODEL_DIR / 'supp_figure_s1_landuse_availability_counts.csv', index=False, encoding='utf-8-sig')
counts.to_csv(MODEL_DIR / 'supp_figure_s1_landuse_resolved_columns.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'source_file':[str(path)]}).to_csv(
    MODEL_DIR / 'supp_figure_s1_landuse_source_file.csv', index=False, encoding='utf-8-sig'
)
df[['grid_id','grid_lat','grid_lon','landuse_status']].to_csv(
    MODEL_DIR / 'supp_figure_s1_landuse_availability_by_grid.csv', index=False, encoding='utf-8-sig'
)


In [ ]:
# ============================================================
# 3. Draw Supplementary Figure S1
# ============================================================

# Natural Earth boundaries are used only as cartographic background.
countries_url = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip'
admin1_url = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip'

world = gpd.read_file(countries_url)
japan = world[world['ADMIN'] == 'Japan'].copy()

try:
    admin1 = gpd.read_file(admin1_url)
    if 'adm0_name' in admin1.columns:
        japan_admin1 = admin1[admin1['adm0_name'] == 'Japan'].copy()
    elif 'admin' in admin1.columns:
        japan_admin1 = admin1[admin1['admin'] == 'Japan'].copy()
    else:
        japan_admin1 = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')
except Exception as e:
    print('Prefecture boundaries could not be loaded; continuing with national outline only:', repr(e))
    japan_admin1 = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

avail = df[df['landuse_available']].copy()
miss = df[~df['landuse_available']].copy()

fig, ax = plt.subplots(figsize=(7.6, 8.6))
japan.plot(ax=ax, color='white', edgecolor='black', linewidth=0.8, zorder=1)
if len(japan_admin1) > 0:
    japan_admin1.boundary.plot(ax=ax, color='0.78', linewidth=0.35, zorder=2)

ax.scatter(
    avail['grid_lon'], avail['grid_lat'], marker='s', s=13, alpha=0.55,
    label=f'Land-use available (n={len(avail):,})', zorder=3
)
ax.scatter(
    miss['grid_lon'], miss['grid_lat'], marker='s', s=17, alpha=0.90,
    label=f'Land-use missing (n={len(miss):,})', zorder=4
)

ax.set_xlim(123.5, 149.0)  # full 5,491-grid study extent
ax.set_ylim(24.0, 46.0)    # full 5,491-grid study extent
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal', adjustable='box')
ax.grid(True, linewidth=0.25, alpha=0.25)
ax.legend(loc='upper left', frameon=True, fontsize=9)
plt.tight_layout()

out_png = MODEL_DIR / 'supp_figure_s1_landuse_data_availability_map.png'
out_pdf = MODEL_DIR / 'supp_figure_s1_landuse_data_availability_map.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()

print('Saved:', out_png)
print('Saved:', out_pdf)


## Suggested caption

**Supplementary Figure S1.** Spatial distribution of 10-km grid cells with and without available land-use composition indicators. Land-use composition indicators were calculated from readable representative points in the National Land Numerical Information detailed land-use mesh data within 5 km of each grid centroid. A grid cell was classified as available when all five composition indicators (waterbody, urban, paddy field, farmland, and forest) were available. Cells without readable land-use representative points within the 5-km search radius are shown as missing. These missing values were retained in the strict Extra Trees analysis and imputed using medians estimated from the corresponding training data.
